# 🚀 ColabMCP v2.0 - AiTun Free Tunnel Edition

[English](colab_mcp_server_en.ipynb) | [中文](colab_mcp_server.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ctz168/colabmcp/blob/main/colab_mcp_server_en.ipynb)

This notebook launches a remote execution server that lets you run code remotely from your local machine using the **ColabMCP client**.

## ✨ What's New in v2.0
- 🌐 **Switched to AiTun** - uses [aitun.cc](https://aitun.cc) no-sign-up tunnel, no ngrok token needed
- 🆓 **No-registration mode** - zero config by default, auto-assigned `https://aitun.cc/XXXXXXXX` public URL
- 🏷️ **Optional registered mode** - set token + subdomain for a fixed subdomain `https://myname.t.aitun.cc`
- 🛡️ **Tunnel watchdog** - auto-reconnects when the subprocess dies; restarts the tunnel when the local service is unresponsive
- 🧠 **Single namespace** - variables persist across requests; even on exception, defined variables are saved
- ⏹️ **Interrupt execution** - interrupt running code without stopping the server
- 📊 **State tracking** - real-time view of current directory and execution state
- 📜 **Command history** - view past executions
- 🌊 **Streaming output** - SSE real-time push of stdout/stderr, ideal for long-running tasks
- ⏱️ **Extended timeout** - default 600s, max 1800s

---
## 📋 Prerequisites

**No-registration mode**: nothing needed, just run all cells.

**Registered mode (optional, for a fixed subdomain)**: register at [aitun.cc](https://aitun.cc) to get a token.

## 1️⃣ Install Dependencies

Installs Flask / requests / psutil, and auto-downloads the aitun-client binary (from the official aitun.cc).

In [ ]:
# Install Python deps
!pip install flask requests psutil -q

# Install aitun-client (official install script with SHA256 verification)
import subprocess, os, shutil

if not shutil.which('aitun'):
    print('⏳ Installing aitun-client...')
    subprocess.run(
        ['bash', '-c', 'curl -fsSL https://aitun.cc/install.sh | bash'],
        check=True
    )
else:
    print('✅ aitun-client already installed')

# Verify installation
r = subprocess.run(['aitun', '--help'], capture_output=True, text=True)
print(r.stdout.split('\n')[0] if r.stdout else r.stderr)
print('✅ Dependencies installed!')

## 2️⃣ Configure Tunnel (Optional)

**No-registration mode (default)**: just skip this cell and run the next one. You'll get a public URL like `https://aitun.cc/UKMA9N10`.

**Registered mode**: fill in your aitun.cc token and subdomain prefix below to get a fixed `https://yourname.t.aitun.cc` URL.

In [ ]:
# ===== Tunnel configuration =====

# AiTun auth token (empty = no-registration mode, auto-assigned URL)
# Registered users can get a token at https://aitun.cc
AITUN_TOKEN = ""

# Subdomain prefix (registered mode only; empty = use token's default subdomain)
AITUN_SUBDOMAIN = ""

# AiTun server (usually no need to change)
AITUN_SERVER = "aitun.cc:6639"

# Local Flask server port
SERVER_PORT = 5000

# ===== Config summary =====
if AITUN_TOKEN:
    print(f"🔐 Registered mode")
    if AITUN_SUBDOMAIN:
        print(f"🏷️  Subdomain: https://{AITUN_SUBDOMAIN}.t.aitun.cc")
    else:
        print(f"🏷️  Using token's default subdomain")
else:
    print(f"🆓 No-registration mode (auto-assigned https://aitun.cc/XXXXXXXX)")
print(f"🔌 Local port: {SERVER_PORT}")
print(f"🌐 AiTun server: {AITUN_SERVER}")

## 3️⃣ Write the Server Code (v2.0.0)

Writes the ColabMCP v2.0.0 server code to `colab_server.py`.

In [ ]:
%%writefile colab_server.py
import os
import sys
import json
import time
import traceback
import subprocess
import gc
import threading
import signal
import ctypes
import queue
import re
import logging
from datetime import datetime
from io import StringIO
from flask import Flask, request, jsonify, Response
import psutil

# ============== Flask 日志降噪 ==============
# 把 werkzeug/flask 的日志 handler 绑定到真正的 stderr，避免被 /execute_stream 的 StringIO 捕获
_REAL_STDERR_FD = sys.stderr
logging.raiseExceptions = False  # 写入失败时静默丢弃，避免污染 SSE 流
_wk_log = logging.getLogger('werkzeug')
_wk_log.handlers = [logging.StreamHandler(_REAL_STDERR_FD)]
_wk_log.setLevel(logging.ERROR)  # 屏蔽 INFO 级访问日志
_wk_log.propagate = False
_fk_log = logging.getLogger('flask.app')
_fk_log.handlers = [logging.StreamHandler(_REAL_STDERR_FD)]
_fk_log.setLevel(logging.WARNING)
_fk_log.propagate = False
_rt_log = logging.getLogger()
_rt_log.handlers = [logging.StreamHandler(_REAL_STDERR_FD)]
_rt_log.setLevel(logging.WARNING)

# ============== 全局状态 ==============
runtime_variables = {}
start_time = time.time()
execution_lock = threading.Lock()
keep_running = True

WORK_DIR_DEFAULT = os.environ.get('COLABMCP_WORKDIR', '/content' if os.path.isdir('/content') else os.getcwd())
execution_state = {
    'current_directory': WORK_DIR_DEFAULT,
    'is_executing': False,
    'last_command': '',
    'last_execution_time': 0,
    'last_error': None,
    'command_history': [],
    'started_at': start_time,
}

current_execution_thread = None
interrupt_requested = False

stream_output_queue = None
stream_active = False

app = Flask(__name__)

_BUILTIN_KEYS = set(dir(__builtins__)) if isinstance(__builtins__, dict) else set(dir(__builtins__))
_BUILTIN_KEYS.update(['__builtins__'])

def _save_variables(namespace, keys_before):
    saved_keys = []
    for key, value in namespace.items():
        if key.startswith('_') or key in _BUILTIN_KEYS:
            continue
        if key in keys_before and namespace[key] is keys_before[key]:
            continue
        try:
            runtime_variables[key] = value
            saved_keys.append(key)
        except Exception:
            pass
    return saved_keys

def heartbeat_thread():
    while keep_running:
        try:
            current_time = time.strftime('%H:%M:%S')
            is_exec = execution_state['is_executing']
            exec_flag = ' [执行中]' if is_exec else ''
            print(f'[心跳] {current_time} - 运行中 | 目录: {execution_state["current_directory"]}{exec_flag}', flush=True)
            time.sleep(60)
        except Exception as e:
            print(f'[心跳错误] {e}', flush=True)
            time.sleep(30)

def _check_gpu():
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False

def _add_to_history(command, output_preview='', success=True):
    entry = {
        'command': command[:500],
        'output_preview': output_preview[:200],
        'timestamp': time.time(),
        'datetime': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'directory': execution_state['current_directory'],
        'success': success
    }
    execution_state['command_history'].append(entry)
    if len(execution_state['command_history']) > 100:
        execution_state['command_history'] = execution_state['command_history'][-100:]

def _update_directory_from_code(code):
    match = re.search(r"os\.chdir\(['\"]([^'\"]+)['\"]\)", code)
    if match:
        execution_state['current_directory'] = match.group(1)
        return match.group(1)
    match = re.search(r"(?:^|\n)%cd\s+(.+?)(?:\n|$)", code)
    if match:
        new_dir = match.group(1).strip().strip('\"\'')
        execution_state['current_directory'] = new_dir
        return new_dir
    return None

def _interrupt_thread(thread):
    global interrupt_requested
    interrupt_requested = True
    if thread and thread.is_alive():
        try:
            thread_id = thread.ident
            if thread_id:
                exc = KeyboardInterrupt()
                ctypes.pythonapi.PyThreadState_SetAsyncExc(
                    ctypes.c_long(thread_id),
                    ctypes.py_object(exc)
                )
        except Exception as e:
            print(f'[中断] 尝试中断失败: {e}', flush=True)
    return True

@app.route('/', methods=['GET'])
def index():
    return jsonify({
        'name': 'ColabMCP Server',
        'version': '2.0.0',
        'status': 'running',
        'uptime_minutes': round((time.time() - start_time) / 60, 2),
        'current_directory': execution_state['current_directory'],
        'is_executing': execution_state['is_executing'],
        'variables_count': len(runtime_variables),
        'endpoints': ['/health', '/probe', '/execute', '/execute_stream', '/interrupt', '/status', '/history', '/variables', '/files', '/cleanup']
    })

@app.route('/health', methods=['GET'])
def health_check():
    mem = psutil.virtual_memory()
    return jsonify({
        'status': 'ok',
        'uptime_minutes': round((time.time() - start_time) / 60, 2),
        'memory_available_gb': round(mem.available / (1024**3), 2),
        'memory_total_gb': round(mem.total / (1024**3), 2),
        'memory_used_pct': round(mem.percent, 2),
        'gpu_available': _check_gpu(),
        'current_directory': execution_state['current_directory'],
        'is_executing': execution_state['is_executing'],
        'variables_count': len(runtime_variables)
    })

@app.route('/status', methods=['GET'])
def get_status():
    return jsonify({
        'status': 'ok',
        'current_directory': execution_state['current_directory'],
        'is_executing': execution_state['is_executing'],
        'last_command': execution_state['last_command'],
        'last_execution_time': execution_state['last_execution_time'],
        'last_error': execution_state['last_error'],
        'recent_history': [h['command'] for h in execution_state['command_history'][-5:]],
        'variables_count': len(runtime_variables),
        'uptime_minutes': round((time.time() - start_time) / 60, 2)
    })

@app.route('/history', methods=['GET'])
def get_history():
    limit = request.args.get('limit', 20, type=int)
    limit = min(limit, 100)
    history = execution_state['command_history'][-limit:]
    return jsonify({'history': history, 'total': len(execution_state['command_history'])})

@app.route('/interrupt', methods=['POST'])
def interrupt_execution():
    global interrupt_requested, current_execution_thread
    if not execution_state['is_executing']:
        return jsonify({'success': True, 'message': '当前没有正在执行的任务'})
    interrupt_requested = True
    if current_execution_thread and current_execution_thread.is_alive():
        success = _interrupt_thread(current_execution_thread)
        if success:
            execution_state['is_executing'] = False
            execution_state['last_error'] = '用户中断'
            return jsonify({'success': True, 'message': '已发送中断信号'})
        else:
            return jsonify({'success': False, 'message': '中断失败，请稍后重试'})
    return jsonify({'success': True, 'message': '中断请求已处理'})

@app.route('/probe', methods=['GET'])
def probe_environment():
    gpu_info = ''
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'],
                              capture_output=True, text=True, timeout=10)
        gpu_info = result.stdout
    except Exception:
        gpu_info = 'No GPU available'
    installed_packages = []
    try:
        result = subprocess.run(['pip', 'list', '--format=freeze'], capture_output=True, text=True, timeout=30)
        for line in result.stdout.split('\n'):
            if '==' in line:
                installed_packages.append(line.strip())
    except Exception:
        pass
    mem = psutil.virtual_memory()
    return jsonify({
        'gpu_info': gpu_info,
        'memory_total_gb': round(mem.total / (1024**3), 2),
        'memory_available_gb': round(mem.available / (1024**3), 2),
        'python_version': sys.version,
        'current_directory': execution_state['current_directory'],
        'installed_packages': installed_packages[:100],
        'total_packages': len(installed_packages)
    })

@app.route('/execute', methods=['POST'])
def execute_code():
    global current_execution_thread, interrupt_requested
    if not execution_lock.acquire(blocking=False):
        return jsonify({'success': False, 'error': '另一个代码正在执行中，请稍后重试'})
    interrupt_requested = False
    current_execution_thread = threading.current_thread()
    try:
        data = request.get_json()
        code = data.get('code', '')
        timeout = min(data.get('timeout', 600), 1800)
        if not code:
            return jsonify({'success': False, 'error': 'No code provided'})
        execution_state['is_executing'] = True
        execution_state['last_command'] = code[:200] + '...' if len(code) > 200 else code
        exec_namespace = {'__builtins__': __builtins__, **runtime_variables}
        keys_before = dict(exec_namespace)
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = captured_stdout = StringIO()
        sys.stderr = captured_stderr = StringIO()
        start_exec_time = time.time()
        saved_keys = []
        try:
            if interrupt_requested:
                raise KeyboardInterrupt('执行被用户中断')
            exec(code, exec_namespace)
            if interrupt_requested:
                raise KeyboardInterrupt('执行被用户中断')
            saved_keys = _save_variables(exec_namespace, keys_before)
            _update_directory_from_code(code)
            stdout_val = captured_stdout.getvalue()
            _add_to_history(code, stdout_val, success=True)
            execution_state['last_execution_time'] = time.time() - start_exec_time
            execution_state['last_error'] = None
            return jsonify({
                'success': True,
                'stdout': stdout_val,
                'stderr': captured_stderr.getvalue(),
                'execution_time_sec': round(time.time() - start_exec_time, 3),
                'variables': saved_keys,
                'current_directory': execution_state['current_directory']
            })
        except KeyboardInterrupt:
            saved_keys = _save_variables(exec_namespace, keys_before)
            stdout_val = captured_stdout.getvalue()
            _add_to_history(code, stdout_val, success=False)
            execution_state['last_error'] = '用户中断'
            return jsonify({
                'success': False,
                'error': '执行被用户中断',
                'error_type': 'KeyboardInterrupt',
                'stdout': stdout_val,
                'stderr': captured_stderr.getvalue(),
                'execution_time_sec': round(time.time() - start_exec_time, 3),
                'variables_saved': saved_keys
            })
        except Exception as e:
            saved_keys = _save_variables(exec_namespace, keys_before)
            stdout_val = captured_stdout.getvalue()
            _add_to_history(code, stdout_val, success=False)
            execution_state['last_error'] = str(e)
            return jsonify({
                'success': False,
                'error': str(e),
                'error_type': type(e).__name__,
                'traceback': traceback.format_exc(),
                'stdout': stdout_val,
                'stderr': captured_stderr.getvalue(),
                'execution_time_sec': round(time.time() - start_exec_time, 3),
                'variables_saved': saved_keys
            })
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr
            execution_state['is_executing'] = False
    except Exception as e:
        execution_state['is_executing'] = False
        return jsonify({
            'success': False,
            'error': f'服务器内部错误: {str(e)}',
            'error_type': type(e).__name__,
            'traceback': traceback.format_exc()
        })
    finally:
        execution_lock.release()
        current_execution_thread = None

@app.route('/execute_stream', methods=['POST'])
def execute_code_stream():
    global stream_output_queue, stream_active, interrupt_requested
    def generate_sse(output_queue):
        try:
            while True:
                try:
                    msg = output_queue.get(timeout=0.5)
                    if msg is None:
                        break
                    yield f'data: {json.dumps(msg, ensure_ascii=False)}\n\n'
                except queue.Empty:
                    yield f': heartbeat\n\n'
                    continue
        except GeneratorExit:
            pass
    stream_output_queue = queue.Queue()
    stream_active = True
    interrupt_requested = False
    data = request.get_json()
    code = data.get('code', '')
    timeout = min(data.get('timeout', 600), 1800)
    if not code:
        stream_output_queue.put({'type': 'error', 'content': 'No code provided'})
        stream_output_queue.put(None)
        return Response(generate_sse(stream_output_queue), mimetype='text/event-stream')
    execution_state['is_executing'] = True
    execution_state['last_command'] = code[:200] + '...' if len(code) > 200 else code
    stripped_code = code.strip()
    shell_match = re.match(r'^import subprocess; result = subprocess\.run\([\'"](.+?)[\'"], shell=True', stripped_code)
    if shell_match:
        shell_cmd = shell_match.group(1)
        def run_shell_command():
            global stream_active
            start_time_exec = time.time()
            stream_output_queue.put({'type': 'status', 'content': f'执行: {shell_cmd}'})
            try:
                process = subprocess.Popen(
                    shell_cmd, shell=True,
                    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                    text=True, bufsize=1,
                    cwd=execution_state['current_directory']
                )
                import select
                while True:
                    if interrupt_requested:
                        process.terminate()
                        stream_output_queue.put({'type': 'error', 'content': '执行被用户中断'})
                        break
                    retcode = process.poll()
                    read_ready, _, _ = select.select([process.stdout, process.stderr], [], [], 0.1)
                    for stream in read_ready:
                        if stream == process.stdout:
                            line = process.stdout.readline()
                            if line:
                                stream_output_queue.put({'type': 'stdout', 'content': line})
                        elif stream == process.stderr:
                            line = process.stderr.readline()
                            if line:
                                stream_output_queue.put({'type': 'stderr', 'content': line})
                    if retcode is not None:
                        remaining_stdout, remaining_stderr = process.communicate()
                        if remaining_stdout:
                            stream_output_queue.put({'type': 'stdout', 'content': remaining_stdout})
                        if remaining_stderr:
                            stream_output_queue.put({'type': 'stderr', 'content': remaining_stderr})
                        break
                elapsed = time.time() - start_time_exec
                stream_output_queue.put({'type': 'complete', 'content': f'完成 (退出码: {process.returncode}, 耗时: {elapsed:.2f}s)'})
                _add_to_history(code, f'shell: {shell_cmd}', success=True)
                execution_state['last_execution_time'] = elapsed
            except Exception as e:
                stream_output_queue.put({'type': 'error', 'content': str(e)})
                _add_to_history(code, str(e), success=False)
            finally:
                stream_output_queue.put(None)
                stream_active = False
                execution_state['is_executing'] = False
        thread = threading.Thread(target=run_shell_command, daemon=True)
        thread.start()
    else:
        class StreamingOutput:
            def __init__(self, q, stream_type):
                self.queue = q
                self.stream_type = stream_type
                self.buffer = []
            def write(self, text):
                if text:
                    self.buffer.append(text)
                    self.queue.put({'type': self.stream_type, 'content': text})
            def flush(self):
                pass
            def getvalue(self):
                return ''.join(self.buffer)
        def run_python_code():
            global stream_active
            start_time_exec = time.time()
            stream_output_queue.put({'type': 'status', 'content': '执行 Python 代码...'})
            old_stdout = sys.stdout
            old_stderr = sys.stderr
            stdout_capture = StreamingOutput(stream_output_queue, 'stdout')
            stderr_capture = StreamingOutput(stream_output_queue, 'stderr')
            sys.stdout = stdout_capture
            sys.stderr = stderr_capture
            exec_namespace = {'__builtins__': __builtins__, **runtime_variables}
            keys_before = dict(exec_namespace)
            saved_keys = []
            try:
                if interrupt_requested:
                    raise KeyboardInterrupt('执行被用户中断')
                exec(code, exec_namespace)
                if interrupt_requested:
                    raise KeyboardInterrupt('执行被用户中断')
                saved_keys = _save_variables(exec_namespace, keys_before)
                _update_directory_from_code(code)
                elapsed = time.time() - start_time_exec
                stream_output_queue.put({'type': 'complete', 'content': f'完成 (耗时: {elapsed:.2f}s)', 'variables': saved_keys})
                _add_to_history(code, ''.join(stdout_capture.buffer)[:200], success=True)
                execution_state['last_execution_time'] = elapsed
            except KeyboardInterrupt:
                saved_keys = _save_variables(exec_namespace, keys_before)
                stream_output_queue.put({'type': 'error', 'content': '执行被用户中断', 'variables_saved': saved_keys})
                _add_to_history(code, '中断', success=False)
            except Exception as e:
                saved_keys = _save_variables(exec_namespace, keys_before)
                stream_output_queue.put({'type': 'error', 'content': f'错误: {type(e).__name__}: {str(e)}', 'variables_saved': saved_keys})
                _add_to_history(code, str(e), success=False)
            finally:
                sys.stdout = old_stdout
                sys.stderr = old_stderr
                stream_output_queue.put(None)
                stream_active = False
                execution_state['is_executing'] = False
        thread = threading.Thread(target=run_python_code, daemon=True)
        thread.start()
    return Response(generate_sse(stream_output_queue), mimetype='text/event-stream')

@app.route('/variables', methods=['GET'])
def list_variables():
    vars_info = {}
    for key, value in runtime_variables.items():
        try:
            var_info = {'type': str(type(value).__name__)}
            if hasattr(value, 'shape'):
                var_info['shape'] = list(value.shape) if hasattr(value.shape, '__iter__') else str(value.shape)
            if hasattr(value, '__len__'):
                try:
                    var_info['length'] = len(value)
                except Exception:
                    pass
            vars_info[key] = var_info
        except Exception:
            vars_info[key] = {'type': str(type(value).__name__)}
    return jsonify({'variables': vars_info, 'count': len(vars_info), 'current_directory': execution_state['current_directory']})

@app.route('/files', methods=['GET'])
def list_files():
    content_dir = execution_state.get('current_directory', WORK_DIR_DEFAULT)
    dir_param = request.args.get('dir', None)
    if dir_param:
        content_dir = dir_param
    files = []
    try:
        for f in os.listdir(content_dir):
            path = os.path.join(content_dir, f)
            try:
                size = os.path.getsize(path)
                files.append({
                    'name': f,
                    'path': path,
                    'size_bytes': size,
                    'size_readable': f'{size/1024:.1f} KB' if size < 1024*1024 else f'{size/1024/1024:.1f} MB',
                    'is_dir': os.path.isdir(path)
                })
            except Exception:
                pass
    except Exception as e:
        return jsonify({'error': str(e), 'files': [], 'directory': content_dir})
    return jsonify({'files': files, 'count': len(files), 'directory': content_dir})

@app.route('/cleanup', methods=['POST'])
def cleanup():
    global runtime_variables
    runtime_variables = {}
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    mem = psutil.virtual_memory()
    return jsonify({'success': True, 'message': 'Memory cleaned', 'memory_available_gb': round(mem.available / (1024**3), 2)})

def signal_handler(sig, frame):
    global keep_running
    print('\n[信号] 收到停止信号，正在关闭...')
    keep_running = False
    sys.exit(0)

if __name__ == '__main__':
    signal.signal(signal.SIGINT, signal_handler)
    signal.signal(signal.SIGTERM, signal_handler)
    port = int(os.environ.get('PORT', 5000))
    print('\n' + '='*60)
    print('🚀 ColabMCP 服务器启动中...')
    print('='*60)
    print(f'版本: 2.0.0 (稳定版)')
    print(f'端口: {port}')
    print(f'工作目录: {WORK_DIR_DEFAULT}')
    print('功能: 心跳保活 + 错误隔离 + 中断支持 + 状态跟踪 + 流式输出')
    print('核心: 单一命名空间 exec() - 变量跨请求持久化')
    print('优化: 异常时也保存变量 + 长任务稳定支持 + 600s超时')
    print('='*60 + '\n')
    heartbeat = threading.Thread(target=heartbeat_thread, daemon=True)
    heartbeat.start()
    try:
        app.run(port=port, host='0.0.0.0', threaded=True, use_reloader=False)
    except Exception as e:
        print(f'[错误] Flask 启动失败: {e}')
        raise

print('✅ 服务器代码已保存到 colab_server.py')

## 4️⃣ Write the AiTun Tunnel Watchdog

Writes aitun_tunnel.py to the working directory. This module:
- Auto-installs aitun-client (if not already installed)
- Starts the aitun-client subprocess and parses the public URL
- Auto-reconnects when the subprocess dies (exponential backoff)
- Periodically probes the local service; triggers a tunnel restart if unresponsive
- Cleans up the subprocess on main-process exit

In [ ]:
%%writefile aitun_tunnel.py
"""aitun_tunnel.py — AiTun 隧道守护器（来自 colabmcp 仓库）"""
from __future__ import annotations
import os, re, sys, time, shutil, signal, subprocess, threading, platform
import urllib.request, urllib.error
from typing import Optional, Callable, List

AITUN_BIN_NAME = 'aitun'
AITUN_DOWNLOAD_BASE = 'https://aitun.cc/downloads'
AITUN_INSTALL_SCRIPT = 'https://aitun.cc/install.sh'
DEFAULT_SERVER = 'aitun.cc:6639'
MAX_STARTUP_WAIT = 60
RECONNECT_DELAY = 3
RECONNECT_BACKOFF_MAX = 30
HEALTH_PROBE_TIMEOUT = 5
HEALTH_PROBE_INTERVAL = 30

def detect_platform() -> str:
    os_name = platform.system().lower()
    arch = platform.machine().lower()
    if os_name == 'linux': platform_os = 'linux'
    elif os_name == 'darwin': platform_os = 'darwin'
    elif os_name in ('mingw', 'msys', 'cygwin', 'windows'): platform_os = 'windows'
    else: platform_os = 'linux'
    if arch in ('x86_64', 'amd64'): platform_arch = 'amd64'
    elif arch in ('aarch64', 'arm64'): platform_arch = 'arm64'
    else: platform_arch = 'amd64'
    return f'{platform_os}-{platform_arch}'

def find_aitun_binary() -> Optional[str]:
    found = shutil.which(AITUN_BIN_NAME)
    if found: return found
    for p in ['/usr/local/bin/aitun', os.path.expanduser('~/.local/bin/aitun'), '/usr/bin/aitun']:
        if os.path.isfile(p) and os.access(p, os.X_OK): return p
    return None

def install_aitun(force: bool = False, verbose: bool = True) -> str:
    if not force:
        existing = find_aitun_binary()
        if existing:
            if verbose: print(f'[aitun] 已安装: {existing}')
            return existing
    if verbose: print('[aitun] 通过官方 install.sh 安装...')
    try:
        result = subprocess.run(['bash', '-c', f'curl -fsSL {AITUN_INSTALL_SCRIPT} | bash'],
                                capture_output=True, text=True, timeout=120)
        if result.returncode == 0:
            installed = find_aitun_binary()
            if installed:
                if verbose: print(f'[aitun] 安装成功: {installed}')
                return installed
            for p in ['/usr/local/bin/aitun', os.path.expanduser('~/.local/bin/aitun'), '/usr/bin/aitun']:
                if os.path.isfile(p) and os.access(p, os.X_OK):
                    if verbose: print(f'[aitun] 安装成功: {p}')
                    return p
        else:
            if verbose: print(f'[aitun] install.sh 失败: {result.stderr[:300]}')
    except Exception as e:
        if verbose: print(f'[aitun] install.sh 异常: {e}')
    if verbose: print('[aitun] 尝试直接下载二进制...')
    plat = detect_platform()
    binary_name = f'aitun-client-{plat}'
    url = f'{AITUN_DOWNLOAD_BASE}/{binary_name}'
    target_dir = os.path.expanduser('~/.local/bin')
    os.makedirs(target_dir, exist_ok=True)
    target = os.path.join(target_dir, AITUN_BIN_NAME)
    try:
        if verbose: print(f'[aitun] 下载 {url} -> {target}')
        urllib.request.urlretrieve(url, target)
        os.chmod(target, 0o755)
        return target
    except Exception as e:
        raise RuntimeError(f'无法安装 aitun-client: {e}')

def parse_public_url(line: str) -> Optional[str]:
    m = re.search(r'https?://(?:[\w.-]+\.)?aitun\.cc/[A-Za-z0-9]+', line)
    if m: return m.group(0).rstrip('.,;"\'')
    return None

def parse_tunnel_code(line: str) -> Optional[str]:
    m = re.search(r'\[OK\]\s*Tunnel code:\s*([A-Za-z0-9]+)', line)
    if m: return m.group(1)
    return None

def probe_local_health(port: int, timeout: float = HEALTH_PROBE_TIMEOUT) -> bool:
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=timeout) as resp:
            return resp.status == 200
    except Exception:
        return False

class AiTunTunnel:
    def __init__(self, local_port: int, local_host: str = 'localhost',
                 token: Optional[str] = None, subdomain: Optional[str] = None,
                 no_p2p: bool = True, server: str = DEFAULT_SERVER,
                 binary: Optional[str] = None, auto_install: bool = True,
                 on_url: Optional[Callable[[str], None]] = None,
                 on_log: Optional[Callable[[str], None]] = None,
                 on_reconnect: Optional[Callable[[int, str], None]] = None,
                 verbose: bool = True):
        self.local_port = local_port
        self.local_host = local_host
        self.token = token or ''
        self.subdomain = subdomain
        self.no_p2p = no_p2p
        self.server = server
        self.binary = binary
        self.auto_install = auto_install
        self.on_url = on_url
        self.on_log = on_log
        self.on_reconnect = on_reconnect
        self.verbose = verbose
        self._proc = None
        self._public_url = None
        self._tunnel_code = None
        self._stop_event = threading.Event()
        self._reader_thread = None
        self._watchdog_thread = None
        self._health_thread = None
        self._reconnect_count = 0
        self._lock = threading.Lock()

    def _log(self, msg: str):
        line = f'[aitun] {msg}'
        if self.verbose: print(line, flush=True)
        if self.on_log:
            try: self.on_log(line)
            except Exception: pass

    def _ensure_binary(self) -> str:
        if self.binary and os.path.isfile(self.binary): return self.binary
        existing = find_aitun_binary()
        if existing:
            self.binary = existing
            return existing
        if not self.auto_install:
            raise RuntimeError('未找到 aitun-client，且 auto_install=False')
        self.binary = install_aitun(verbose=self.verbose)
        return self.binary

    def _build_cmd(self) -> List[str]:
        cmd = [self.binary, '-p', str(self.local_port), '-host', self.local_host, '-s', self.server]
        if self.token: cmd.extend(['-k', self.token])
        if self.subdomain and self.token: cmd.extend(['--subdomain', self.subdomain])
        if self.no_p2p: cmd.append('--no-p2p')
        return cmd

    def _reader_loop(self):
        proc = self._proc
        if proc is None or proc.stdout is None: return
        for line in iter(proc.stdout.readline, ''):
            if not line: break
            line_stripped = line.rstrip()
            if line_stripped: self._log(f'[client] {line_stripped}')
            url = parse_public_url(line)
            if url:
                with self._lock:
                    if self._public_url != url:
                        self._public_url = url
                        self._log(f'📡 公网 URL: {url}')
                        if self.on_url:
                            try: self.on_url(url)
                            except Exception as e: self._log(f'on_url 回调异常: {e}')
            code = parse_tunnel_code(line)
            if code:
                with self._lock:
                    self._public_url = self._public_url or f'https://aitun.cc/{code}'
                    self._tunnel_code = code

    def _health_loop(self):
        while not self._stop_event.is_set():
            try:
                time.sleep(HEALTH_PROBE_INTERVAL)
                if self._stop_event.is_set(): break
                ok = probe_local_health(self.local_port)
                if not ok:
                    self._log('⚠️ 本地服务无响应，触发隧道重连...')
                    self._kill_proc()
            except Exception as e:
                self._log(f'健康探测异常: {e}')

    def _watchdog_loop(self):
        delay = RECONNECT_DELAY
        while not self._stop_event.is_set():
            try:
                proc = self._proc
                if proc is None: break
                while proc.poll() is None and not self._stop_event.is_set():
                    time.sleep(1.0)
                if self._stop_event.is_set(): break
                rc = proc.returncode
                self._reconnect_count += 1
                reason = f'子进程退出 (rc={rc})'
                self._log(f'⚠️ {reason}，{delay}s 后重启 (第 {self._reconnect_count} 次)...')
                if self.on_reconnect:
                    try: self.on_reconnect(self._reconnect_count, reason)
                    except Exception: pass
                if self._stop_event.wait(delay): break
                delay = min(delay + 2, RECONNECT_BACKOFF_MAX)
                with self._lock:
                    self._public_url = None
                    self._tunnel_code = None
                self._spawn_proc()
                delay = RECONNECT_DELAY
            except Exception as e:
                self._log(f'看门狗异常: {e}')
                time.sleep(RECONNECT_DELAY)

    def _spawn_proc(self):
        cmd = self._build_cmd()
        self._log(f'启动: {" ".join(cmd)}')
        try:
            self._proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                         text=True, bufsize=1, start_new_session=True)
        except FileNotFoundError:
            self._log('❌ 找不到 aitun 二进制，尝试重新安装...')
            self.binary = None
            self._ensure_binary()
            self._proc = subprocess.Popen(self._build_cmd(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                         text=True, bufsize=1, start_new_session=True)
        self._reader_thread = threading.Thread(target=self._reader_loop, name='aitun-reader', daemon=True)
        self._reader_thread.start()

    def _kill_proc(self):
        proc = self._proc
        if proc is None: return
        try:
            if proc.poll() is None:
                try: os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
                except (ProcessLookupError, PermissionError, OSError): proc.terminate()
                try: proc.wait(timeout=3)
                except subprocess.TimeoutExpired:
                    try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
                    except (ProcessLookupError, PermissionError, OSError): proc.kill()
                    proc.wait(timeout=2)
        except Exception as e:
            self._log(f'终止子进程异常: {e}')

    def start(self, block: bool = True):
        self._ensure_binary()
        self._spawn_proc()
        self._watchdog_thread = threading.Thread(target=self._watchdog_loop, name='aitun-watchdog', daemon=True)
        self._watchdog_thread.start()
        self._health_thread = threading.Thread(target=self._health_loop, name='aitun-health', daemon=True)
        self._health_thread.start()
        try: signal.signal(signal.SIGTERM, self._signal_handler)
        except (ValueError, OSError): pass
        if block:
            try:
                while not self._stop_event.is_set(): time.sleep(1.0)
            except KeyboardInterrupt: self._log('收到中断信号，停止...')
            finally: self.stop()

    def stop(self):
        self._stop_event.set()
        self._kill_proc()
        self._log('已停止')

    def _signal_handler(self, signum, frame):
        self._log(f'收到信号 {signum}，停止隧道...')
        self.stop()

    @property
    def public_url(self) -> Optional[str]:
        with self._lock: return self._public_url

    @property
    def tunnel_code(self) -> Optional[str]:
        with self._lock: return self._tunnel_code

    @property
    def is_running(self) -> bool:
        return self._proc is not None and self._proc.poll() is None

    def wait_for_url(self, timeout: float = MAX_STARTUP_WAIT) -> Optional[str]:
        start = time.time()
        while time.time() - start < timeout:
            url = self.public_url
            if url: return url
            if self._stop_event.is_set(): break
            time.sleep(0.3)
        return self.public_url

print('✅ aitun_tunnel.py 已保存')

## 5️⃣ Start the Flask Server in the Background

Starts Flask in a background thread so the same notebook can continue to launch the tunnel.

In [ ]:
import subprocess, os, time, signal

# Kill any leftover stale processes
os.system('pkill -f colab_server.py 2>/dev/null || true')
time.sleep(1)

# Start Flask in the background
env = {**os.environ, 'PORT': str(SERVER_PORT)}
server_proc = subprocess.Popen(
    ['python3', 'colab_server.py'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Wait for the server to be ready
import urllib.request, json
for i in range(20):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{SERVER_PORT}/health', timeout=2) as r:
            data = json.loads(r.read())
            print(f'✅ Flask server ready (port={SERVER_PORT})')
            print(f'   Status: {data}')
            break
    except Exception as e:
        time.sleep(0.5)
else:
    print('❌ Flask server failed to start')
    # Print last 30 lines of log
    server_proc.terminate()
    out, _ = server_proc.communicate(timeout=5)
    print(out[-3000:])
    raise RuntimeError('Flask not ready')

## 6️⃣ Start the AiTun Tunnel (foreground, blocking — keep running)

**Important**: keep this cell running after execution, do not stop it!

You will see:
- `[aitun] [client] [OK] Proxy URL: https://aitun.cc/XXXXXXXX` — this is your public URL
- `[aitun] [心跳] HH:MM:SS - 运行中...` — service is healthy
- If the tunnel drops unexpectedly, the watchdog auto-reconnects

If you accidentally stop this cell, just run it again — it will auto-reconnect.

In [ ]:
import sys, time
sys.path.insert(0, '.')
from aitun_tunnel import AiTunTunnel

def on_url(url):
    print('\n' + '='*70)
    print('🎉 AiTun tunnel established!')
    print('='*70)
    print(f'\n📡 Public URL: {url}')
    print(f'\n📋 Copy this URL for client connection:')
    print(f'   {url}')
    print('\n' + '-'*70)
    print('🔧 Available endpoints:')
    print(f'   Health check:    GET  {url}/health')
    print(f'   Probe env:       GET  {url}/probe')
    print(f'   Execute code:    POST {url}/execute')
    print(f'   Stream exec:     POST {url}/execute_stream')
    print(f'   Interrupt:       POST {url}/interrupt')
    print(f'   Get status:      GET  {url}/status')
    print(f'   Get history:     GET  {url}/history')
    print(f'   List variables:  GET  {url}/variables')
    print(f'   List files:      GET  {url}/files')
    print(f'   Cleanup memory:  POST {url}/cleanup')
    print('-'*70)
    print('\n⚠️  Keep this cell running, or the tunnel will drop')
    print('='*70 + '\n', flush=True)

def on_reconnect(attempt, reason):
    print(f'\n🔄 [Reconnect] #{attempt}: {reason}\n', flush=True)

tunnel = AiTunTunnel(
    local_port=SERVER_PORT,
    token=AITUN_TOKEN or None,
    subdomain=AITUN_SUBDOMAIN or None,
    server=AITUN_SERVER,
    no_p2p=True,  # P2P off is recommended for Colab
    verbose=True,
    on_url=on_url,
    on_reconnect=on_reconnect,
)

# Blocking; exits gracefully on Ctrl+C or cell stop
tunnel.start(block=True)

---
## ⚠️ Tips to Prevent Colab from Disconnecting

### Method 1: keep the browser tab active
- Don't background the browser tab
- Click on the Colab page periodically

### Method 2: use an anti-sleep script
Run this in the browser console (F12):
```javascript
function KeepAlive() {
    console.log('keeping alive...');
    document.querySelector('colab-connect-button').click();
}
setInterval(KeepAlive, 60000);
```

---
## 📝 Client Usage

### Python client
```bash
pip install requests
```

```python
import requests

url = 'https://aitun.cc/XXXXXXXX'  # replace with your URL

# Health check
print(requests.get(f'{url}/health').json())

# Execute code (variables persist across requests)
print(requests.post(f'{url}/execute', json={'code': 'x = 42'}).json())
print(requests.post(f'{url}/execute', json={'code': 'print(x*2)'}).json())  # outputs 84

# Streaming execution (ideal for long tasks)
import requests
with requests.post(f'{url}/execute_stream', json={'code': 'import time\nfor i in range(5):\n    print(i)\n    time.sleep(1)'}, stream=True) as r:
    for line in r.iter_lines():
        if line and line.startswith(b'data: '):
            print(line[6:].decode())
```

### CLI client (using this repo's client/colab_client.py)
```bash
python client/colab_client.py --url https://aitun.cc/XXXXXXXX --health
python client/colab_client.py --url https://aitun.cc/XXXXXXXX -e "print('Hello!')"
python client/colab_client.py --url https://aitun.cc/XXXXXXXX -i
```